In [ ]:
# =============================================================================
# NOTEBOOK: wf_ctd_build (v1.0.0)
# PROJECT: Sea of Cortez Advanced Analytical Pipeline (Baja 2026)
# PURPOSE: Ingest .cnv files, apply TEOS-10 physics, and hydrate DuckDB
# =============================================================================

import pandas as pd
import numpy as np
import duckdb
import pathlib
import re
import gsw
from scipy.signal import savgol_filter
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, module='gsw.stability')

# ==========================================
# 1. CONFIGURATION & LOOKUP TABLES
# ==========================================
INPUT_PATH = pathlib.Path(r'C:\Users\joeac\Jupyter\WF_CTD\cnv')
DATABASE_PATH = pathlib.Path.cwd() / "processed" / "sea_of_cortez.duckdb"

STATION_MAP = {
    'cast4':   'PuntaMarciel_Deep_West',    
    'cast5':   'IslaCalavera_North',        
    'OSU1970': 'Historical_Ref_Station',   
    'PP500J':  'PrimaryProd_J_500m',
    'PP500K':  'PrimaryProd_K_150m'
}

GPS_LOOKUP = {
    'PuntaMarciel':           (23.9915, -109.8210),
    'PuntaMarciel_Deep_West': (23.9950, -109.8500),
    'IslaCalavera_North':     (26.6150, -111.8600),
    'IslaTiburon':            (28.9821, -112.4203),
    'CanalDeBallenas':        (29.0515, -113.3120),
    'Mulege':                 (26.8904, -111.9551),
    'DeepChannel':            (25.5620, -110.1250),
    'SanFrancisquito':        (28.4350, -112.8710),
    'Monserrat':              (25.6833, -111.0500),
    'SantaCatalina':          (25.6167, -110.7833),
    'PP40':                   (24.3210, -110.3500),
    'PP130':                  (24.3550, -110.4200),
    'DEFAULT':                (24.5800, -110.2100)
}

# PIPELINE CONSTANTS
MIN_SAFE_PRESSURE = 5.0  
AI_WINDOW = 15           
AI_POLY = 3              
OUTPUT_RES = 0.1         

# SENSOR ALIGNMENT & FILTERING
C_SHIFT = 1              
O2_SHIFT = 8             
MIN_DESCENT_RATE = 0.01  

# ==========================================
# 2. MODULE 1: INGESTION
# ==========================================
def ingest_sbe_cnv(file_path):
    columns = []
    start_time_base = None
    recovered_name = None
    data_start_line = 0
    
    try:
        with open(file_path, 'r', errors='ignore') as f:
            header_lines = f.readlines()
            
        for idx, line in enumerate(header_lines):
            if "* FileName =" in line:
                full_path = line.split('=')[-1].strip()
                fname_with_ext = pathlib.Path(full_path).name
                clean_name = fname_with_ext.split('.')[0]
                name_parts = clean_name.split('_')
                recovered_name = name_parts[1] if len(name_parts) > 1 else name_parts[0]

            if "# start_time =" in line:
                ts_str = line.split('=')[-1].split('[')[0].strip()
                try: start_time_base = pd.to_datetime(ts_str)
                except: start_time_base = None

            if line.startswith("# name"):
                label = line.split('=')[1].split(':')[0].strip()
                if re.search(r'tv290C|temp', label, re.I): target = 'temp_raw'
                elif re.search(r'c0S/m|cond', label, re.I): target = 'cond_raw'
                elif re.search(r'prdM|pres', label, re.I): target = 'pres_raw'
                elif re.search(r'sbeox0Mm/L', label, re.I): target = 'o2_umol_l'
                elif re.search(r'ph', label, re.I): target = 'ph_raw'
                elif re.search(r'flECO-AFL|chl', label, re.I): target = 'chl_raw'
                elif re.search(r'timeS', label, re.I): target = 'time_elapsed'
                else: target = label
                columns.append(target)
                
            if "*END*" in line:
                data_start_line = idx + 1; break

        df = pd.read_csv(file_path, skiprows=data_start_line, sep=r'\s+', names=columns, engine='python')
        
        final_id = recovered_name if recovered_name else file_path.stem.split('_')[0]
        station_clean = STATION_MAP.get(final_id, final_id)
        df['station_name'] = station_clean
        lat, lon = GPS_LOOKUP.get(station_clean, GPS_LOOKUP['DEFAULT'])
        df['lat'], df['lon'] = lat, lon
        
        cast_id = 1
        nums = re.findall(r'cast(\d+)', file_path.stem, re.I)
        if nums: cast_id = int(nums[0])
        df['cast_no'] = cast_id
        df['station_id'] = f"{station_clean}_C{cast_id}"
        
        df['time_iso'] = start_time_base + pd.to_timedelta(df['time_elapsed'], unit='s') if start_time_base else pd.Timestamp('2026-01-01')

        return df
    except Exception as e:
        print(f"⚠️ Error parsing {file_path.name}: {e}")
        return None

# ==========================================
# 3. ANALYTICAL PROCESSING
# ==========================================
print(f"🚀 Initializing wf_ctd_build v1.0.0...")
all_files = list(INPUT_PATH.glob("*.cnv"))
valid_dfs = []

for f in all_files:
    raw_df = ingest_sbe_cnv(f)
    if raw_df is None: continue
    
    # SENSOR ALIGNMENT
    if 'cond_raw' in raw_df.columns: raw_df['cond_raw'] = raw_df['cond_raw'].shift(-C_SHIFT)
    if 'o2_umol_l' in raw_df.columns: raw_df['o2_umol_l'] = raw_df['o2_umol_l'].shift(-O2_SHIFT)

    # RULE 1: PRESSURE UNIQUENESS
    raw_df = raw_df.drop_duplicates(subset=['pres_raw'])

    # RULE 2: LOOP FILTER & CLIP
    raw_df['dp'] = raw_df['pres_raw'].diff()
    temp_df = raw_df[(raw_df['pres_raw'] > MIN_SAFE_PRESSURE) & (raw_df['dp'] > MIN_DESCENT_RATE)].copy()
    temp_df = temp_df.dropna().reset_index(drop=True)
    if len(temp_df) < AI_WINDOW: continue

    # RULE 3: TEOS-10 PHYSICS
    lat_val, lon_val = temp_df['lat'].iloc[0], temp_df['lon'].iloc[0]
    
    temp_df['SP'] = gsw.SP_from_C(temp_df['cond_raw']*10, temp_df['temp_raw'], temp_df['pres_raw'])
    temp_df['SA'] = gsw.SA_from_SP(temp_df['SP'], temp_df['pres_raw'], lon_val, lat_val)
    temp_df['CT'] = gsw.CT_from_t(temp_df['SA'], temp_df['temp_raw'], temp_df['pres_raw'])
    temp_df['rho'] = gsw.rho(temp_df['SA'], temp_df['CT'], temp_df['pres_raw'])
    temp_df['sigma0'] = gsw.sigma0(temp_df['SA'], temp_df['CT'])
    
    # N-Squared calculation
    n2, p_mid = gsw.Nsquared(temp_df['SA'], temp_df['CT'], temp_df['pres_raw'], lat=lat_val)
    temp_df['n2'] = np.append(n2, n2[-1]) 
    
    # Spiciness & Sound Speed
    temp_df['spice'] = gsw.spiciness0(temp_df['SA'], temp_df['CT'])
    temp_df['sound_speed'] = gsw.sound_speed(temp_df['SA'], temp_df['CT'], temp_df['pres_raw'])
    
    # Biogeochemicals
    if 'o2_umol_l' in temp_df.columns: temp_df['o2_final'] = temp_df['o2_umol_l'] * (1000 / temp_df['rho'])
    if 'chl_raw' in temp_df.columns: temp_df['chl_final'] = ((temp_df['chl_raw'] - 0.013) * 0.26).clip(lower=0)
    if 'ph_raw' in temp_df.columns: temp_df['ph_final'] = temp_df['ph_raw']

    # QUALITY CONTROL (QC) FLAGS
    temp_df['qc_flag'] = 1  
    temp_df.loc[temp_df['sigma0'].diff() < -0.03, 'qc_flag'] = 3
    temp_df.loc[(temp_df['temp_raw'] < -2.0) | (temp_df['temp_raw'] > 35.0), 'qc_flag'] = 4
    temp_df.loc[(temp_df['SP'] < 2.0) | (temp_df['SP'] > 42.0), 'qc_flag'] = 4

    # RULE 4: SILK-SMOOTHING
    targets = [c for c in ['CT', 'SA', 'SP', 'o2_final', 'chl_final', 'ph_final', 'sigma0', 'n2', 'spice', 'sound_speed'] if c in temp_df.columns]
    for col in targets:
        temp_df[col] = temp_df[col].replace([np.inf, -np.inf], np.nan).interpolate().bfill().ffill()
        temp_df[col] = savgol_filter(temp_df[col], AI_WINDOW, AI_POLY)
    
    valid_dfs.append(temp_df)
    print(f"✅ Processed: {f.name} | Observations: {len(temp_df)}")

# ==========================================
# 4. WAREHOUSING
# ==========================================
if valid_dfs:
    df = pd.concat(valid_dfs, ignore_index=True)
    df['dbar_bin'] = (df['pres_raw'] / OUTPUT_RES).round() * OUTPUT_RES
    
    agg_config = {
        'time_iso': 'min', 'CT': 'mean', 'SA': 'mean', 'SP': 'mean', 'o2_final': 'mean', 
        'chl_final': 'mean', 'ph_final': 'mean', 'sigma0': 'mean', 'n2': 'mean', 
        'spice': 'mean', 'sound_speed': 'mean', 'lat': 'first', 'lon': 'first',
        'qc_flag': 'max' 
    }
    actual_agg = {k: v for k, v in agg_config.items() if k in df.columns}
    
    df_bin = df.groupby(['station_id', 'station_name', 'cast_no', 'dbar_bin']).agg(actual_agg).reset_index()
    df_bin['depth_m'] = gsw.z_from_p(df_bin['dbar_bin'], df_bin['lat']) * -1

    with duckdb.connect(str(DATABASE_PATH)) as db:
        db.execute("DROP TABLE IF EXISTS wf_ctd_binned")
        db.execute("CREATE TABLE wf_ctd_binned AS SELECT * FROM df_bin")
    
    print("-" * 65)
    print(f"🏁 wf_ctd_build v1.0.0 SUCCESS | Table 'wf_ctd_binned' Hydrated.")
    print("-" * 65)